# Atlas DeePC Reacher: fair-data subspace success

This notebook records a successful selected-chart subspace DeePC case for `Reacher-v5`.

The pipeline is:

1. Collect a fair offline dataset using Select-DPC-style IID random input trajectories.
2. Build a Hankel matrix from the offline trajectories.
3. Partition the Hankel columns into multiple charts using the prediction-error atlas method.
4. Compress each chart with SVD and run DeePC on the selected chart subspace.
5. Verify that the fingertip stabilizes at the target.

The offline data collection follows the Select-DPC Reacher setup: IID inputs, 200 simulations, 200 steps each. No PD/expert rollout and no target-specific oversampling are used.


In [ ]:
# =========================
# 0) Imports and paths
# =========================
from pathlib import Path
import json
import os

PROJECT_DIR = Path.cwd().resolve()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_DIR / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_DIR / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

import importlib
import expert_pipeline_search as pipe
import targeted_search as select_data

# Reload in case the notebook kernel cached older helper modules.
pipe = importlib.reload(pipe)
select_data = importlib.reload(select_data)
collect_select_deepc_iid = select_data.collect_select_deepc_iid

HANKEL_PATH = PROJECT_DIR / "select_iid_offline_hankel.npy"
LABEL_PATH = PROJECT_DIR / "select_iid_prediction_error_labels.npz"
RESULT_PATH = PROJECT_DIR / "select_iid_subspace_control_results.npz"
VIDEO_PATH = PROJECT_DIR / "select_iid_subspace_control.mp4"
FIGURE_PATH = PROJECT_DIR / "select_iid_subspace_tracking.png"
XY_VIDEO_PATH = PROJECT_DIR / "select_iid_subspace_xy_trajectory.mp4"

print("Project:", PROJECT_DIR)
print("Hankel path:", HANKEL_PATH)


In [ ]:
# =========================
# 1) Offline data collection
# =========================
# Set FORCE_RECOLLECT=True to regenerate the Select-DPC-style IID offline dataset.
# This follows the Select-DPC Reacher protocol: IID inputs, no expert policy, no test-target oversampling.
FORCE_RECOLLECT = False

if FORCE_RECOLLECT or not HANKEL_PATH.exists():
    H = collect_select_deepc_iid(seed=2026, episodes=200, episode_len=200, input_scale=1.0)
else:
    H = np.load(HANKEL_PATH)

print("H shape:", H.shape)
print("input range:", H[:pipe.M * pipe.L].min(), H[:pipe.M * pipe.L].max())


In [ ]:
# =========================
# 2) Prediction-error atlas learning
# =========================
# This keeps the method: KMeans initialization in trajectory-feature space,
# then label refinement by local ridge-predictor prediction error.
FORCE_RELEARN_ATLAS = False

if FORCE_RELEARN_ATLAS or not LABEL_PATH.exists():
    labels = pipe.learn_prediction_error_atlas(H, seed=7, label_path=LABEL_PATH)
else:
    labels = np.load(LABEL_PATH)["labels"]

print("K:", pipe.K)
print("cluster sizes:", np.bincount(labels, minlength=pipe.K).tolist())


## Prediction-Error Subspace Charts

The fair IID offline data is kept fixed. After prediction-error labels are learned, each chart is compressed by SVD and DeePC is solved on a chart subspace.

For chart `k`, with cluster matrix `H_k`, the linear subspace is:

```text
H_k ~= U_k[:, :r_k] S_k V_k^T
trajectory_window = U_k[:, :r_k] alpha
```

The successful fair-data case uses `T_ini = 6`, `T_f = 6`, online chart selection among the 8 atlas charts, and a rank-90 SVD basis. It runs for 420 steps and remains inside the 2 cm target neighborhood over the final 50 steps, so it is not just stopping at the crossing point.


In [ ]:
# =========================
# 3a) Build SVD subspaces for each prediction-error chart
# =========================
# The fair IID Hankel windows have length 12. The successful controller uses
# more past samples and a shorter future horizon than the first prototype.
CONTROL_T_INI = 6
CONTROL_T_F = 6
pipe.T_INI = CONTROL_T_INI
pipe.T_F = CONTROL_T_F
pipe.L = CONTROL_T_INI + CONTROL_T_F

SUBSPACE_RANK = 90
SUBSPACE_ENERGY = None
subspace_bases = pipe.compute_bases(H, labels, rank=SUBSPACE_RANK, energy=SUBSPACE_ENERGY)
subspace_ranks = [0 if basis is None else basis.shape[1] for basis in subspace_bases]

np.savez(
    PROJECT_DIR / "prediction_error_subspaces_rank90.npz",
    ranks=np.asarray(subspace_ranks, dtype=int),
    labels=labels,
    rank=np.array(SUBSPACE_RANK),
    t_ini=np.array(CONTROL_T_INI),
    t_f=np.array(CONTROL_T_F),
)

print("control horizon:", {"T_ini": pipe.T_INI, "T_f": pipe.T_F, "L": pipe.L})
print("subspace rank cap:", SUBSPACE_RANK)
print("subspace ranks by chart:", subspace_ranks)
print("cluster sizes:", np.bincount(labels, minlength=pipe.K).tolist())


In [ ]:
# =========================
# 3b) Selected-chart subspace atlas DeePC controller
# =========================
# Successful fair-data case found in the sandbox:
#   IID offline data -> prediction-error atlas -> online selected chart -> rank-90 SVD basis -> DeePC.
variant = pipe.Variant(
    name="subspace_rank90_chart2",
    mode="subspace",
    rank=SUBSPACE_RANK,
    energy=SUBSPACE_ENERGY,
    max_cols=500,
    try_top=8,
    u_limit=0.35,
    q_xy=200000.0,
    q_dq=300.0,
    q_joint=0.0,
    q_sum=0.0,
    r=0.01,
    lam_g=0.1,
    lam_ini=100000.0,
    lam_du=10.0,
    solver="CLARABEL",
    steps=420,
    warmup_scale=0.0,
    seed=1,
    chart_selection="best_predicted",
    fixed_chart=None,
    use_ik_ref=False,
)

print("active variant:", variant.name)
print("active mode:", variant.mode)
print("fixed atlas chart:", variant.fixed_chart)
print("online chart selection:", variant.fixed_chart is None)


In [ ]:
# =========================
# 4) Run selected-chart subspace atlas DeePC
# =========================
# This one line asks the MuJoCo Reacher environment to render rgb_array frames
# and save the simulation video while running the controller.
summary = pipe.run_atlas_deepc(
    H,
    labels,
    variant,
    save_path=RESULT_PATH,
    video_path=VIDEO_PATH,
)
summary


In [ ]:
# =========================
# 5) Verify subspace stabilization and save tracking diagnostics plot
# =========================
data = np.load(RESULT_PATH, allow_pickle=True)
y = np.asarray(data["y"], dtype=float)
ref = np.asarray(data["ref"], dtype=float)
u = np.asarray(data["u"], dtype=float)
y_pred = np.asarray(data["pred"], dtype=float) if "pred" in data.files else np.full_like(y, np.nan)
g_norm = np.asarray(data["g_norm"], dtype=float) if "g_norm" in data.files else np.full(y.shape[0], np.nan)
dist = np.asarray(data["distance"], dtype=float)

print("steps:", dist.size)
print("first distance:", dist[0])
print("minimum distance:", dist.min())
print("last distance:", dist[-1])
print("mean last 30:", dist[-30:].mean())
print("max last 30:", dist[-30:].max())
print("unique selected charts:", np.unique(data["chart"]).astype(int).tolist())

success_mean = dist[-30:].mean() < 0.01
success_max = dist[-30:].max() < 0.02
success = bool(success_mean and success_max)
print("success criteria passed:", success)

lengths = [y.shape[0], ref.shape[0], u.shape[0], dist.shape[0], y_pred.shape[0], g_norm.shape[0]]
horizon = min(lengths)
t = np.arange(horizon)
y = y[:horizon]
ref = ref[:horizon]
u = u[:horizon]
y_pred = y_pred[:horizon]
g_norm = g_norm[:horizon]

fig, axes = plt.subplots(6, 1, figsize=(12, 17), sharex=True)
fig.suptitle("Reacher tracking diagnostics", fontsize=16)

plot_specs = [
    (4, "dq1", "dq1 meas", "dq1 pred", "dq1 ref"),
    (5, "dq2", "dq2 meas", "dq2 pred", "dq2 ref"),
    (6, "x_ee", "x_ee meas", "x_ee pred", "x_ee target"),
    (7, "y_ee", "y_ee meas", "y_ee pred", "y_ee target"),
]

for ax, (channel, ylabel, meas_label, pred_label, ref_label) in zip(axes[:4], plot_specs):
    ax.plot(t, y[:, channel], label=meas_label)
    ax.plot(t, y_pred[:, channel], "--", label=pred_label)
    ax.plot(t, ref[:, channel], ":", linewidth=1.8, label=ref_label)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[4].plot(t, u[:, 0], label="u1")
axes[4].plot(t, u[:, 1], label="u2")
axes[4].axhline(variant.u_limit, color="red", linestyle="--", linewidth=1.0, alpha=0.65, label="upper limit")
axes[4].axhline(-variant.u_limit, color="red", linestyle="--", linewidth=1.0, alpha=0.65, label="lower limit")
axes[4].set_ylabel("input")
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="best")

if np.isfinite(g_norm).any():
    axes[5].plot(t, g_norm, label="||g||")
else:
    axes[5].plot(t, np.zeros_like(t), label="||g|| not saved")
axes[5].set_ylabel("||g||")
axes[5].set_xlabel("Time step")
axes[5].grid(True, alpha=0.3)
axes[5].legend(loc="best")

fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.show()
print("Saved tracking diagnostics figure:", FIGURE_PATH)
if success:
    print("Success: selected-chart atlas DeePC stabilizes the fingertip at the target.")
else:
    print("This run did not meet the strict stabilization threshold, but diagnostics were saved for inspection.")


In [ ]:
# =========================
# 6) Save lightweight XY trajectory video
# =========================
# Optional diagnostic video showing the fingertip trace relative to the target.
import imageio.v2 as imageio

xy = y[:, 6:8]
target = ref[0, 6:8]
frames = []
frame_indices = list(range(0, len(xy), 3))
if frame_indices[-1] != len(xy) - 1:
    frame_indices.append(len(xy) - 1)

for idx in frame_indices:
    fig, ax = plt.subplots(figsize=(5, 5), dpi=120)
    ax.plot(xy[:idx + 1, 0], xy[:idx + 1, 1], color="tab:blue", linewidth=2)
    ax.scatter([xy[0, 0]], [xy[0, 1]], color="tab:gray", s=35, label="start")
    ax.scatter([xy[idx, 0]], [xy[idx, 1]], color="tab:blue", s=45, label="fingertip")
    ax.scatter([target[0]], [target[1]], color="tab:green", marker="*", s=160, label="target")
    ax.add_patch(plt.Circle(target, 0.02, color="tab:green", fill=False, linestyle="--", alpha=0.6))
    ax.set_xlim(-0.22, 0.22)
    ax.set_ylim(-0.22, 0.22)
    ax.set_aspect("equal", "box")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    ax.set_title(f"step {idx} | distance {dist[idx]:.4f}")
    ax.set_xlabel("x_ee")
    ax.set_ylabel("y_ee")
    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy())
    plt.close(fig)

imageio.mimsave(XY_VIDEO_PATH, frames, fps=20, macro_block_size=16)
print("Saved MuJoCo simulation video:", VIDEO_PATH)
print("Saved XY trajectory video:", XY_VIDEO_PATH)


## Notes

The offline data is collected with IID random inputs following the Select-DPC Reacher ablation, not with IK-PD and not with seed-42 oversampling. During control, actions are produced by DeePC over a rank-90 SVD subspace basis from the online-selected prediction-error atlas chart.

The successful run uses the same strict threshold as before: `mean_last_30 < 0.01` and `max_last_30 < 0.02`. In the sandbox run, the result was `mean_last_30 = 0.00793`, `max_last_30 = 0.01243`, `tail50_mean = 0.00701`, `tail50_max = 0.01243`, and `last = 0.00502`.
